# BPM phage genome promoter scanner

目的：用資料夾內的 `BPM` core promoter scoring model 掃描 phage genome，找出高分的潛在宿主 RNAP promoter。

這份 notebook 不依賴已知 TSS。它會在兩股上枚舉 `-35 + spacer + -10` core promoter window，計算 BPM 分數，並輸出可調 threshold 的候選清單。

重要設定：
- `SPACER_LENGTHS`: BPM 目前支援 15..19 bp spacer。
- `MIN_LOGEXP`: 分數 cutoff，目前是 `2.0`。要重新決定門檻就先設成 `None`，看
  `score_summary` 的分位數與 top hits，再填回數字重跑。
- `ACCESSIONS`: `None` 代表掃 `data/phage/*.gb` 全部；也可指定清單。

> 記憶體：cutoff 是在**每個 genome 的掃描迴圈內**就套用的。17 個 genome 共 1.32 Mbp，
> 兩股 × 五種 spacer = 13.2 M 個 window，全部留成 DataFrame 會吃掉約 8 GB、concat 當下
> 再翻倍。現在只有過關的 window 會變成 row，分位數所需的分布另外用純 float 累積，
> 尖峰記憶體由單一 genome 決定。


In [1]:
# === Path bootstrap (shared by 01-07) ===
# Locates MS2_Data_PyTorch/scripts/library_release by walking up from the cwd,
# then imports _paths, which sets every other path absolutely and puts
# MS2_Data_PyTorch/scripts on sys.path. Safe to run from any working directory.
import sys
from pathlib import Path

for _c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _rel = _c / "MS2_Data_PyTorch" / "scripts" / "library_release"
    if (_rel / "_paths.py").exists():
        if str(_rel) not in sys.path:
            sys.path.insert(0, str(_rel))
        break
else:
    raise RuntimeError(f"library_release not found from {Path.cwd()}")

from _paths import *  # noqa: F401,F403

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("RELEASE_OUT  :", RELEASE_OUT)


PROJECT_ROOT : C:\project\Whole-model
DATA_DIR     : C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\data
RELEASE_OUT  : C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\outputs


In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq

# Paths come from _paths (bootstrap cell above): genomes in data/phage,
# results in outputs/. BPM lives in MS2_Data_PyTorch/scripts, which _paths
# already put on sys.path, so `from BPM import BPM` resolves to the one
# canonical copy rather than a notebook-local duplicate.
DATA = PHAGE_DIR
OUT = RELEASE_OUT

from BPM import BPM as bpm



In [ ]:
# === 掃描設定 ===
# None = 掃 data/phage/*.gb 全部；或改成例如 ['AY543070.1', 'NC_001604.1']
ACCESSIONS = None

# BPM.py 的 spacer energy 目前定義 15, 16, 17, 18, 19
SPACER_LENGTHS = [15, 16, 17, 18, 19]

# 分數 cutoff。None = 不套 cutoff，改用 TOP_N_PER_GROUP 先看分布和 top hits。
MIN_LOGEXP = 3.0

# cutoff 為 None 時，每個 genome/strand/spacer 保留前 N 個高分片段供檢查
TOP_N_PER_GROUP = 200

# 分數分布摘要要看的分位數
QUANTILES = [0.5, 0.9, 0.95, 0.99, 0.999]

# 輸出檔
OUT_CSV = OUT / '03_phage_scan_candidates_raw.csv'  # intermediate; standardised table is written by the last cell

In [4]:
CANDIDATE_COLS = [
    'accession', 'genome_len', 'strand',
    'genome_start_1based', 'genome_end_1based', 'predicted_tss_1based',
    'spacer_len', 'core_len', 'minus35', 'spacer', 'minus10', 'core_seq',
    'dG_minus35', 'dG_spacer', 'dG_minus10', 'dG_total', 'pred_exp', 'pred_logexp',
]


def gb_files(accessions=None):
    if accessions is None:
        return sorted(DATA.glob('*.gb'))
    files = []
    for acc in accessions:
        p = DATA / f'{acc}.gb'
        if not p.exists():
            raise FileNotFoundError(p)
        files.append(p)
    return files

def iter_core_windows(seq, spacer_lengths):
    seq = str(seq).upper()
    for spacer_len in spacer_lengths:
        core_len = 6 + spacer_len + 6
        for start0 in range(0, len(seq) - core_len + 1):
            core = seq[start0:start0 + core_len]
            if set(core) <= {'A', 'C', 'G', 'T'}:
                yield start0, core_len, spacer_len, core

def tss_pos_from_core(start0, core_len, strand, genome_len):
    # BPM scores only core promoter (-35/spacer/-10). This notebook uses the
    # first transcribed base immediately after the scored core as nominal +1.
    if strand == '+':
        return start0 + core_len + 1
    return genome_len - start0 - core_len

def genomic_span_from_oriented_window(start0, core_len, strand, genome_len):
    if strand == '+':
        return start0 + 1, start0 + core_len
    return genome_len - (start0 + core_len) + 1, genome_len - start0

def score_core(core):
    m35, spacer, m10 = bpm.promoter_elements(core)
    dG35, dGsp, dG10 = bpm.score_promoter_elements(core)
    total_dG = dG35 + dGsp + dG10
    logexp = bpm.score2logexp(total_dG)
    exp = bpm.score2exp(total_dG)
    return m35, spacer, m10, dG35, dGsp, dG10, total_dG, exp, logexp

def scan_record(record, spacer_lengths, min_logexp=None, top_n=None, quantiles=QUANTILES):
    """Scan one genome and return (candidates, score_summary).

    The cutoff is applied here, inside the per-genome loop, rather than on one
    concatenated table afterwards. Two strands x five spacer lengths over the 17
    genomes is 13.2 M windows; keeping every one of them as a row costs ~8 GB and
    the concat that followed doubled it. Only windows that survive the cutoff
    become rows, while the distribution the quantile summary needs is accumulated
    as bare floats (~70 MB for the largest genome). `description` is deliberately
    not stored per row - it is one string per genome, and the caller maps it back
    on after the concat.
    """
    acc = record.id
    seq_plus = str(record.seq).upper()
    genome_len = len(seq_plus)
    strands = [('+', seq_plus), ('-', str(Seq(seq_plus).reverse_complement()))]

    rows = []
    scores = {}   # (strand, spacer_len) -> list[float], for the quantile table
    for strand, oriented_seq in strands:
        for start0, core_len, spacer_len, core in iter_core_windows(oriented_seq, spacer_lengths):
            m35, spacer, m10, dG35, dGsp, dG10, total_dG, exp, logexp = score_core(core)
            scores.setdefault((strand, spacer_len), []).append(logexp)
            if min_logexp is not None and logexp < min_logexp:
                continue
            g_start, g_end = genomic_span_from_oriented_window(start0, core_len, strand, genome_len)
            rows.append({
                'accession': acc,
                'genome_len': genome_len,
                'strand': strand,
                'genome_start_1based': g_start,
                'genome_end_1based': g_end,
                'predicted_tss_1based': tss_pos_from_core(start0, core_len, strand, genome_len),
                'spacer_len': spacer_len,
                'core_len': core_len,
                'minus35': m35,
                'spacer': spacer,
                'minus10': m10,
                'core_seq': core,
                'dG_minus35': dG35,
                'dG_spacer': dGsp,
                'dG_minus10': dG10,
                'dG_total': total_dG,
                'pred_exp': exp,
                'pred_logexp': logexp,
            })

    candidates = pd.DataFrame(rows, columns=CANDIDATE_COLS)
    del rows
    if min_logexp is None and top_n is not None:
        candidates = (
            candidates
            .sort_values('pred_logexp', ascending=False)
            .groupby(['strand', 'spacer_len'], as_index=False)
            .head(top_n)
        )

    summary = pd.DataFrame([
        {'accession': acc, 'strand': strand, 'spacer_len': spacer_len,
         'n_windows': len(values),
         **{q: float(np.quantile(values, q)) for q in quantiles}}
        for (strand, spacer_len), values in sorted(scores.items())
    ])
    return candidates.reset_index(drop=True), summary

In [5]:
# === 執行掃描 ===
# 每個 genome 掃完就立刻套 cutoff 並丟掉沒過關的 window，所以尖峰記憶體由最大的
# 那個 genome 決定，不是由全部 13.2 M 個 window 決定。
frames, summaries, desc_by_acc = [], [], {}
for gb in gb_files(ACCESSIONS):
    record = SeqIO.read(gb, 'genbank')
    desc_by_acc[record.id] = record.description
    cand_i, summary_i = scan_record(record, SPACER_LENGTHS,
                                    min_logexp=MIN_LOGEXP, top_n=TOP_N_PER_GROUP)
    n_windows = int(summary_i['n_windows'].sum()) if len(summary_i) else 0
    print(f'scanning {record.id}: {len(record.seq):,} bp  ->  '
          f'{n_windows:,} windows, {len(cand_i):,} kept')
    frames.append(cand_i)
    summaries.append(summary_i)
    del record, cand_i, summary_i

candidates = (
    pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=CANDIDATE_COLS)
)
candidates.insert(1, 'description', candidates['accession'].map(desc_by_acc))
score_summary = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
del frames, summaries

total_windows = int(score_summary['n_windows'].sum()) if len(score_summary) else 0
print()
print(f'total scanned windows: {total_windows:,}')
print(f'candidates kept: {len(candidates):,}')
candidates.head()

scanning AY543070.1: 121,750 bp  ->  1,217,220 windows, 1,718 kept


scanning EU568876.1: 41,901 bp  ->  418,730 windows, 6 kept


scanning KC787107.1: 39,318 bp  ->  392,900 windows, 3 kept


scanning NC_000866.4: 168,903 bp  ->  1,687,880 windows, 2,836 kept


scanning NC_001335.1: 52,297 bp  ->  522,690 windows, 32 kept


...

scanning NC_024147.1: 57,315 bp  ->  572,870 windows, 21 kept


scanning NC_047864.1: 38,209 bp  ->  381,810 windows, 150 kept


scanning NC_054907.1: 168,697 bp  ->  1,686,390 windows, 2,852 kept


scanning NC_054931.1: 163,825 bp  ->  1,637,970 windows, 2,705 kept

total scanned windows: 13,213,880
candidates kept: 12,852


,accession,description,genome_len,strand,genome_start_1based,genome_end_1based,predicted_tss_1based,spacer_len,core_len,minus35,spacer,minus10,core_seq,dG_minus35,dG_spacer,dG_minus10,dG_total,pred_exp,pred_logexp
0,AY543070.1,"Escherichia phage T5, complete genome",121750,+,421,448,449,16,28,TAGCCC,AAAATAGTAACTTTTA,TAAACT,TAGCCCAAAATAGTAACTTTTATAAACT,-2.940787,0.71134,-4.688452,-6.917899,353.227908,2.548055
1,AY543070.1,"Escherichia phage T5, complete genome",121750,+,1213,1240,1241,16,28,TTGAAA,TCTAATAGCTCATCAA,TAACGT,TTGAAATCTAATAGCTCATCAATAACGT,-3.574372,0.71134,-3.253528,-6.116560,131.178327,2.117862
2,AY543070.1,"Escherichia phage T5, complete genome",121750,+,1249,1276,1277,16,28,TCGACG,TGATAACGCTGACCAT,TAGAAT,TCGACGTGATAACGCTGACCATTAGAAT,-2.116398,0.71134,-4.750457,-6.155516,138.403862,2.141148
3,AY543070.1,"Escherichia phage T5, complete genome",121750,+,3427,3454,3455,16,28,TTGCCA,AATATAACGCCTTGAA,TAATAG,TTGCCAAATATAACGCCTTGAATAATAG,-3.892034,0.71134,-2.756114,-5.936808,101.983014,2.008528
4,AY543070.1,"Escherichia phage T5, complete genome",121750,+,3546,3573,3574,16,28,TTTGCT,AACTTAGATTTACGTT,TAAAAT,TTTGCTAACTTAGATTTACGTTTAAAAT,-1.679940,0.71134,-5.008389,-5.976989,107.945962,2.033206


In [6]:
# === 分數分布：用來決定 MIN_LOGEXP ===
# 分位數是在套 cutoff 之前、對該 genome 的全部 window 算的，所以即使
# MIN_LOGEXP 已經設好，這張表仍然反映完整分布。
score_summary

,accession,strand,spacer_len,n_windows,0.5,0.9,0.95,0.99,0.999
0,AY543070.1,+,15,121724,0.495250,0.496918,0.501227,0.548666,0.890556
1,AY543070.1,+,16,121723,0.495894,0.549048,0.661856,1.242514,2.134270
2,AY543070.1,+,17,121722,0.497308,0.643973,0.886712,1.657843,2.504008
3,AY543070.1,+,18,121721,0.495667,0.531869,0.616110,1.126289,1.993474
4,AY543070.1,+,19,121720,0.495249,0.496977,0.501395,0.554726,0.928111
...,...,...,...,...,...,...,...,...,...
165,NC_054931.1,-,15,163799,0.495258,0.497350,0.502457,0.558798,0.897216
166,NC_054931.1,-,16,163798,0.496120,0.560832,0.692012,1.322972,2.216350
167,NC_054931.1,-,17,163797,0.498053,0.673867,0.930233,1.739732,2.588772
168,NC_054931.1,-,18,163796,0.495826,0.540010,0.631354,1.134272,1.963260


In [7]:
# === 候選輸出 ===
candidates = (
    candidates
    .sort_values(['accession', 'pred_logexp'], ascending=[True, False])
    .reset_index(drop=True)
)
if MIN_LOGEXP is None:
    print(f'MIN_LOGEXP is None: exported top {TOP_N_PER_GROUP} per accession/strand/spacer group.')
else:
    print(f'exported windows with pred_logexp >= {MIN_LOGEXP}.')

candidates.to_csv(OUT_CSV, index=False)
print('wrote', OUT_CSV, candidates.shape)
candidates.head(30)

exported windows with pred_logexp >= 2.0.
wrote C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\outputs\03_phage_scan_candidates_raw.csv (12852, 19)


,accession,description,genome_len,strand,genome_start_1based,genome_end_1based,predicted_tss_1based,spacer_len,core_len,minus35,spacer,minus10,core_seq,dG_minus35,dG_spacer,dG_minus10,dG_total,pred_exp,pred_logexp
0,AY543070.1,"Escherichia phage T5, complete genome",121750,+,53903,53931,53932,17,29,TTGACA,TCTGCCCTTGAATAAGC,TATAAT,TTGACATCTGCCCTTGAATAAGCTATAAT,-3.937519,0.00000,-5.519893,-9.457412,971.034011,2.987234
1,AY543070.1,"Escherichia phage T5, complete genome",121750,+,66386,66414,66415,17,29,TTGACA,TTTTCGCCGCTTAGGTA,TATACT,TTGACATTTTCGCCGCTTAGGTATATACT,-3.937519,0.00000,-5.199956,-9.137475,952.252311,2.978752
2,AY543070.1,"Escherichia phage T5, complete genome",121750,-,99936,99964,99935,17,29,TTGACA,CAGGTGGAAATTTAGAA,TATACT,TTGACACAGGTGGAAATTTAGAATATACT,-3.937519,0.00000,-5.199956,-9.137475,952.252311,2.978752
3,AY543070.1,"Escherichia phage T5, complete genome",121750,-,21023,21051,21022,17,29,TTGAAA,TTTTCCTCCAAAATTAG,TATAAT,TTGAAATTTTCCTCCAAAATTAGTATAAT,-3.574372,0.00000,-5.519893,-9.094265,948.960162,2.977248
4,AY543070.1,"Escherichia phage T5, complete genome",121750,-,92039,92067,92038,17,29,TTGATA,AAATTTTCCAATACTAT,TATAAT,TTGATAAAATTTTCCAATACTATTATAAT,-3.462334,0.00000,-5.519893,-8.982227,939.397659,2.972849
5,AY543070.1,"Escherichia phage T5, complete genome",121750,-,13238,13266,13237,17,29,TTGACG,ATAATCATATAGTTTGG,TATAAT,TTGACGATAATCATATAGTTTGGTATAAT,-3.458144,0.00000,-5.519893,-8.978037,939.009381,2.972670
6,AY543070.1,"Escherichia phage T5, complete genome",121750,+,66479,66507,66508,17,29,TTGCTA,AATGCTTAAATACTTGC,TATAAT,TTGCTAAATGCTTAAATACTTGCTATAAT,-3.416848,0.00000,-5.519893,-8.936741,935.055747,2.970838
7,AY543070.1,"Escherichia phage T5, complete genome",121750,-,53805,53833,53804,17,29,TTGCTA,TTTTGTACTTAAAATAG,TATAAT,TTGCTATTTTGTACTTAAAATAGTATAAT,-3.416848,0.00000,-5.519893,-8.936741,935.055747,2.970838
8,AY543070.1,"Escherichia phage T5, complete genome",121750,-,50087,50115,50086,17,29,TTGCTA,ATCGCTACCTGATTTGC,TATAAT,TTGCTAATCGCTACCTGATTTGCTATAAT,-3.416848,0.00000,-5.519893,-8.936741,935.055747,2.970838
9,AY543070.1,"Escherichia phage T5, complete genome",121750,-,46012,46040,46011,17,29,TTGCTA,AACGCTTCAAATTCTCG,TATAAT,TTGCTAAACGCTTCAAATTCTCGTATAAT,-3.416848,0.00000,-5.519893,-8.936741,935.055747,2.970838


In [8]:
# === 快速檢視每個 genome 的最高分 hit ===
cols = [
    'accession', 'strand', 'genome_start_1based', 'genome_end_1based',
    'predicted_tss_1based', 'spacer_len', 'minus35', 'minus10',
    'dG_total', 'pred_logexp', 'core_seq'
]
(
    candidates[cols]
    .sort_values(['accession', 'pred_logexp'], ascending=[True, False])
    .groupby('accession', as_index=False)
    .head(20)
)

,accession,strand,genome_start_1based,genome_end_1based,predicted_tss_1based,spacer_len,minus35,minus10,dG_total,pred_logexp,core_seq
0,AY543070.1,+,53903,53931,53932,17,TTGACA,TATAAT,-9.457412,2.987234,TTGACATCTGCCCTTGAATAAGCTATAAT
1,AY543070.1,+,66386,66414,66415,17,TTGACA,TATACT,-9.137475,2.978752,TTGACATTTTCGCCGCTTAGGTATATACT
2,AY543070.1,-,99936,99964,99935,17,TTGACA,TATACT,-9.137475,2.978752,TTGACACAGGTGGAAATTTAGAATATACT
3,AY543070.1,-,21023,21051,21022,17,TTGAAA,TATAAT,-9.094265,2.977248,TTGAAATTTTCCTCCAAAATTAGTATAAT
4,AY543070.1,-,92039,92067,92038,17,TTGATA,TATAAT,-8.982227,2.972849,TTGATAAAATTTTCCAATACTATTATAAT
...,...,...,...,...,...,...,...,...,...,...,...
10162,NC_054931.1,-,51813,51840,51812,16,TTGATA,TATAAT,-8.270887,2.919144,TTGATAGCAAAAAGAGCATGCATATAAT
10163,NC_054931.1,+,90993,91021,91022,17,TTGAAA,TATGAT,-8.248235,2.916391,TTGAAATTAACTGGGTTGAACCATATGAT
10164,NC_054931.1,-,142788,142815,142787,16,TTGCTA,TATAAT,-8.225401,2.913530,TTGCTAAATTTATTCCTTCGGGTATAAT
10165,NC_054931.1,-,82480,82508,82479,17,TTGTAA,TACAAT,-8.212722,2.911904,TTGTAAGGAGCTGAGTTACACACTACAAT


## Notes

- `predicted_tss_1based` 是 nominal +1：假設 +1 緊接在 scored core promoter 之後。BPM 本身只對 core promoter 打分，沒有直接驗證 TSS。
- 若要捕捉 host RNAP promoter，後續建議把高分 hits 和 coding direction、intergenic region、已知 early gene annotation 一起過濾。
- 決定 threshold 後，把 `MIN_LOGEXP` 從 `None` 改成數字重跑最後的候選輸出 cell。



## Standardised output for the assembly step

Writes `outputs/03_phage_promoters.csv` with a 75 nt promoter
(-60..+15 around the predicted TSS), matching 04 so the two phage
sources concatenate directly.

In [9]:
# === Standardised phage-scan table (03_phage_promoters.csv) ===
# The scanner only reports the core (-35 + spacer + -10, 27-31 nt). For the
# library we re-slice a fixed -60..+15 window around the predicted TSS, using
# exactly the coordinate convention 04 uses for the curated set, so 03 and 04
# concatenate into one table with the same 75 nt promoter definition.
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq

FULL_WINDOW = (-60, 15)   # same as T5_FULL_WINDOW in 04
# Self-contained: the scan output path is rebuilt here rather than inherited
# from the config cell, so this cell can be re-run on its own.
RAW_CSV = RELEASE_OUT / "03_phage_scan_candidates_raw.csv"
raw = pd.read_csv(require(RAW_CSV, "phage scan candidates"))
print("candidates in:", len(raw))


def coord_to_gi(tss0, strand, c):
    """promoter coord -> 0-based genome index; coord 0 does not exist."""
    if c == 0:
        raise ValueError("promoter coord 0 does not exist")
    offset = (c - 1) if c > 0 else c
    return tss0 + offset if strand == "+" else tss0 - offset


def slice_region(genome, tss0, strand, a, b):
    """Promoter-oriented (5'->3') sequence, coords a<b inclusive."""
    gi_a = coord_to_gi(tss0, strand, a)
    gi_b = coord_to_gi(tss0, strand, b)
    lo, hi = sorted((gi_a, gi_b))
    if lo < 0 or hi >= len(genome):
        raise IndexError("window runs off the end of the genome")
    sub = genome[lo:hi + 1]
    return sub if strand == "+" else str(Seq(sub).reverse_complement())


genomes = {}
for gb in sorted(PHAGE_DIR.glob("*.gb")):
    rec = SeqIO.read(gb, "genbank")
    genomes[rec.id] = str(rec.seq).upper()
print("genomes loaded:", len(genomes))

a, b = FULL_WINDOW
seqs, ok = [], []
for acc, strand, tss1 in zip(raw["accession"], raw["strand"], raw["predicted_tss_1based"]):
    g = genomes.get(acc)
    if g is None:
        seqs.append(pd.NA); ok.append(False); continue
    try:
        seqs.append(slice_region(g, int(tss1) - 1, strand, a, b))
        ok.append(True)
    except (IndexError, ValueError):
        seqs.append(pd.NA); ok.append(False)

phage = raw.rename(columns={"accession": "phage_genome_accession"}).copy()
phage["promoter_sequence"] = seqs
phage["window_ok"] = ok
phage["phage_promoter_window"] = f"{a}..{b}"
phage["method"] = "computational_scan"
print(f"window sliced ok: {sum(ok)} / {len(ok)}")

out = standardize(phage, source="phage", candidate_id="PHG-S", extra_pass=pd.Series(ok))
out.to_csv(RELEASE_OUT / "03_phage_promoters.csv", index=False)

print(f"\nwrote {RELEASE_OUT / '03_phage_promoters.csv'}  {out.shape}")
print("promoter_length:", out.loc[out['qc_pass'], 'promoter_length'].value_counts().to_dict())
print("qc_pass:", int(out["qc_pass"].sum()), "/", len(out))
print(out[STD_COLS].head(3).to_string(index=False))


candidates in: 12852


genomes loaded: 17
window sliced ok: 12847 / 12852



wrote C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\outputs\03_phage_promoters.csv  (12852, 28)
promoter_length: {75: 12841}
qc_pass: 12841 / 12852
candidate_id source                                                           promoter_sequence  promoter_length  alphabet_valid  qc_pass
  PHG-S00001  phage TCAAAATACAACATCTAAGAGAAAAATTATATTGACATCTGCCCTTGAATAAGCTATAATAGTAGTCTTAGTTAG               75            True     True
  PHG-S00002  phage CATCTACCATATCGGAATTATAAAGTGGTTATTGACATTTTCGCCGCTTAGGTATATACTATTATCATTCAGTTG               75            True     True
  PHG-S00003  phage GAAAAACTCGTAGCGGATATAAAAACCGTTATTGACACAGGTGGAAATTTAGAATATACTGTTAGTAAACCTAAT               75            True     True
